# Step 1.3 — LiDAR Parsing (Upgraded) ✅

| | |
|---|---|
| **Input** | `output/step_0/samples_index.json`, `sensor_meta.json` per sample |
| **Outputs** | `output/step_1/lidar/<sample>/lidar_raw.npy` — points + derived fields (N×7) |
| | `output/step_1/lidar/<sample>/lidar_meta.json` — calibration + per-sample stats |
| | `output/step_1/lidar_summary.csv` — one row per sample (fast) |
| | `output/step_1/lidar/<sample>/lidar_points.csv` — optional, full per-point (opt-in, large) |
| **Used by** | Step 2.1 (RANSAC + DBSCAN), Step 3.3 (LiDAR tracking), Step 4.4 (fusion) |

---

### What changed from the original

1. **Path fix** — `Path(info['folder'])` now correctly joins with `STEP0_DIR`.
2. **Added derived fields** — `distance`, `azimuth_deg`, `elevation_deg` per point, computed vectorized with NumPy and appended as extra columns to the saved array (now N×7 instead of N×4). Your dissertation (section 3.1.3) already describes these fields as being computed — this closes the gap between that description and the actual code.
3. **Added CSV export** — a fast per-sample summary CSV by default (num_points, distance/elevation stats). Full per-point CSV is available but off by default since it would be 10+ million rows across all samples — flip `EXPORT_FULL_POINT_CSV = True` below if you specifically want that.
4. **Kept your calibration handling as-is** — it was already correct.

In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Verify config.py exists
# ─────────────────────────────────────────────────────────────────

from pathlib import Path

if not Path("config.py").exists():
    raise FileNotFoundError(
        "config.py not found in this folder. Copy it from the repo root "
        "and set DATA_ROOT to your nuScenes dataset path."
    )

from config import DATA_ROOT, NUSCENES_VERSION, STEP0_DIR, STEP1_DIR

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}")

LIDAR_OUT_DIR = STEP1_DIR / "lidar"
LIDAR_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Set True only if you want a full per-point CSV per sample (large, slow).
# Default False — the per-sample summary CSV covers most needs.
EXPORT_FULL_POINT_CSV = False

print(f"✅ config.py found")
print(f"✅ DATA_ROOT     : {DATA_ROOT}")
print(f"✅ STEP0_DIR     : {STEP0_DIR}")
print(f"✅ LIDAR_OUT_DIR : {LIDAR_OUT_DIR}")
print(f"✅ EXPORT_FULL_POINT_CSV : {EXPORT_FULL_POINT_CSV}")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output
✅ config.py found
✅ DATA_ROOT     : F:\Sensor fusion Research\DATA SET\archive
✅ STEP0_DIR     : F:\Sensor fusion Research\output\step_0
✅ LIDAR_OUT_DIR : F:\Sensor fusion Research\output\step_1\lidar
✅ EXPORT_FULL_POINT_CSV : False


In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Main pipeline: parse LiDAR points + compute derived fields
# ─────────────────────────────────────────────────────────────────

import json
import numpy as np
import pandas as pd
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud
from tqdm import tqdm

with open(STEP0_DIR / "samples_index.json") as f:
    samples_index = json.load(f)

nusc = NuScenes(version=NUSCENES_VERSION, dataroot=str(DATA_ROOT), verbose=False)

summary_rows = []
skipped_samples = []

for sample_id, info in tqdm(samples_index.items(), total=len(samples_index), desc="Parsing LiDAR"):

    # FIXED — join with STEP0_DIR since "folder" is a relative name
    meta_path = STEP0_DIR / info["folder"] / "sensor_meta.json"
    with open(meta_path) as f:
        sensor_meta = json.load(f)

    if "LIDAR_TOP" not in sensor_meta["sensor_channels"]:
        skipped_samples.append(sample_id)
        continue

    lidar_meta = sensor_meta["sensor_channels"]["LIDAR_TOP"]
    lidar_path = DATA_ROOT / lidar_meta["filename"]

    # Read cloud (N,4): x, y, z, intensity — in LiDAR sensor frame
    lidar_pc = LidarPointCloud.from_file(str(lidar_path))
    pts = lidar_pc.points.T   # (N, 4)

    x, y, z, intensity = pts[:, 0], pts[:, 1], pts[:, 2], pts[:, 3]

    # ── Derived fields (vectorized, matches dissertation section 3.1.3) ──
    distance = np.sqrt(x**2 + y**2 + z**2)
    azimuth_deg = np.degrees(np.arctan2(y, x))
    horizontal_dist = np.sqrt(x**2 + y**2)
    elevation_deg = np.degrees(np.arctan2(z, horizontal_dist))

    # Stack into N×7: x, y, z, intensity, distance, azimuth_deg, elevation_deg
    pts_enriched = np.column_stack([x, y, z, intensity, distance, azimuth_deg, elevation_deg])

    sample_out_dir = LIDAR_OUT_DIR / sample_id
    sample_out_dir.mkdir(parents=True, exist_ok=True)

    npy_path = sample_out_dir / "lidar_raw.npy"
    np.save(npy_path, pts_enriched.astype(np.float32))

    # ── Metadata + calibration (kept from original — was already correct) ──
    meta_out = {
        "sample_id": sample_id,
        "num_points": int(pts.shape[0]),
        "source_file": lidar_meta["filename"],
        "columns": ["x", "y", "z", "intensity", "distance", "azimuth_deg", "elevation_deg"],
        "sensor_to_ego_translation": lidar_meta["sensor_to_ego_translation"],
        "sensor_to_ego_rotation": lidar_meta["sensor_to_ego_rotation"],
        "ego_pose": lidar_meta["ego_pose"]
    }
    with open(sample_out_dir / "lidar_meta.json", "w") as f:
        json.dump(meta_out, f, indent=2)

    # ── Per-sample summary row (fast, always written) ──
    summary_rows.append({
        "sample_id": sample_id,
        "num_points": int(pts.shape[0]),
        "distance_mean": round(float(distance.mean()), 3),
        "distance_min": round(float(distance.min()), 3),
        "distance_max": round(float(distance.max()), 3),
        "elevation_mean_deg": round(float(elevation_deg.mean()), 3),
        "intensity_mean": round(float(intensity.mean()), 3),
    })

    # ── Optional full per-point CSV (opt-in — large) ──
    if EXPORT_FULL_POINT_CSV:
        df_points = pd.DataFrame(
            pts_enriched,
            columns=["x", "y", "z", "intensity", "distance", "azimuth_deg", "elevation_deg"]
        )
        df_points.to_csv(sample_out_dir / "lidar_points.csv", index=False)

assert len(summary_rows) > 0, "No LiDAR samples were processed — check DATA_ROOT and paths"

if skipped_samples:
    print(f"⚠️ {len(skipped_samples)} samples skipped (no LIDAR_TOP): {skipped_samples[:5]}...")
else:
    print("✅ No samples skipped — LIDAR_TOP present in all.")

print(f"\n✅ LiDAR parsing complete: {len(summary_rows)} samples processed.")
print(f"📄 Outputs in: {LIDAR_OUT_DIR}")

Parsing LiDAR:   0%|          | 0/404 [00:00<?, ?it/s]

Parsing LiDAR:   0%|          | 1/404 [00:00<01:00,  6.70it/s]

Parsing LiDAR:   1%|▏         | 6/404 [00:00<00:15, 26.37it/s]

Parsing LiDAR:   2%|▏         | 9/404 [00:00<00:19, 19.85it/s]

Parsing LiDAR:   3%|▎         | 12/404 [00:00<00:29, 13.40it/s]

...

Parsing LiDAR:  98%|█████████▊| 396/404 [00:17<00:00, 13.97it/s]

Parsing LiDAR:  99%|█████████▉| 400/404 [00:17<00:00, 17.57it/s]

Parsing LiDAR: 100%|█████████▉| 403/404 [00:18<00:00, 13.21it/s]

Parsing LiDAR: 100%|██████████| 404/404 [00:18<00:00, 22.37it/s]

✅ No samples skipped — LIDAR_TOP present in all.

✅ LiDAR parsing complete: 404 samples processed.
📄 Outputs in: F:\Sensor fusion Research\output\step_1\lidar


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Save the per-sample summary CSV
# ─────────────────────────────────────────────────────────────────

summary_df = pd.DataFrame(summary_rows).sort_values("sample_id").reset_index(drop=True)

summary_path = STEP1_DIR / "lidar_summary.csv"
summary_df.to_csv(summary_path, index=False)

assert len(summary_df) == len(samples_index) - len(skipped_samples), \
    "Summary row count doesn't match expected sample count"

print(f"✅ Summary CSV saved: {summary_path}")
print(f"   {len(summary_df)} rows | columns: {summary_df.columns.tolist()}")
display(summary_df.head())

✅ Summary CSV saved: F:\Sensor fusion Research\output\step_1\lidar_summary.csv
   404 rows | columns: ['sample_id', 'num_points', 'distance_mean', 'distance_min', 'distance_max', 'elevation_mean_deg', 'intensity_mean']


,sample_id,num_points,distance_mean,distance_min,distance_max,elevation_mean_deg,intensity_mean
0,sample_0000,34688,11.471,0.000,102.879,-11.158,19.851
1,sample_0001,34720,11.487,0.001,105.160,-11.194,18.887
2,sample_0002,34720,10.913,0.000,104.693,-11.213,18.509
3,sample_0003,34688,10.124,0.000,105.064,-11.126,17.494
4,sample_0004,34752,9.671,0.000,104.934,-11.125,19.232


In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 — Quick validation: load one sample back and confirm shape
# ─────────────────────────────────────────────────────────────────

check_sample = summary_df.iloc[50]["sample_id"]
check_path = LIDAR_OUT_DIR / check_sample / "lidar_raw.npy"

loaded = np.load(check_path)

assert loaded.shape[1] == 7, f"Expected 7 columns (x,y,z,intensity,distance,azimuth,elevation), got {loaded.shape[1]}"

print(f"✅ Verified {check_sample}: shape {loaded.shape}")
print(f"   Columns: x, y, z, intensity, distance, azimuth_deg, elevation_deg")
print(f"   Distance range: {loaded[:,4].min():.2f}m to {loaded[:,4].max():.2f}m")
print(f"   Azimuth range : {loaded[:,5].min():.1f}° to {loaded[:,5].max():.1f}°")
print(f"   Elevation range: {loaded[:,6].min():.1f}° to {loaded[:,6].max():.1f}°")

✅ Verified sample_0050: shape (34688, 7)
   Columns: x, y, z, intensity, distance, azimuth_deg, elevation_deg
   Distance range: 0.01m to 88.31m
   Azimuth range : -180.0° to 180.0°
   Elevation range: -51.9° to 10.8°
